In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import json
import pandas as pd

# Load dataset
try:
    with open("dataset_full.json", "r", encoding="utf-8") as f:
        data = json.load(f)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Attempting to load with 'latin-1' encoding.")
    with open("dataset_full.json", "r", encoding="latin-1") as f:
        data = json.load(f)

df = pd.DataFrame(data)

df.head()

,id,entity,key,original_value,mutated_value,value_type,ner_type,topic,claim,label,transformation,detail,evidence,ce_overlap
0,1,अंकित राजपूत,पूरा नाम,अंकित सिंह राजपूत,अंकित सिंह राजपूत,person,PERSON,sports,अंकित राजपूत के पूरा नाम के तौर पर अंकित सिंह ...,SUPPORTS,original,,अंकित राजपूत के उपोद्घात के रूप में गेंदबाज जा...,0.114650
1,2,अंकित राजपूत,पूरा नाम,अंकित सिंह राजपूत,अजीत भालचन्द्र अगरकर,person,PERSON,sports,अजीत भालचन्द्र अगरकर को अंकित राजपूत का पूरा न...,REFUTES,entity_swap,swap 'अंकित सिंह राजपूत'→'अजीत भालचन्द्र अगरकर',अंकित राजपूत के उपोद्घात के रूप में गेंदबाज जा...,0.096573
2,3,अंकित राजपूत,पूरा नाम,अंकित सिंह राजपूत,अनिरुद्ध जोशी,person,PERSON,sports,अनिरुद्ध जोशी ही अंकित राजपूत के पूरा नाम हैं।,REFUTES,paraphrase_corruption,paraphrase+swap 'अंकित सिंह राजपूत'→'अनिरुद्ध ...,अंकित राजपूत के उपोद्घात के रूप में गेंदबाज जा...,0.069841
3,4,अंकित राजपूत,जन्म,"4 दिसम्बर 1993 (आयु 32)कानपुर, उत्तर प्रदेश, भारत","4 दिसम्बर 1993 (आयु 32)कानपुर, उत्तर प्रदेश, भारत",date,OTHER,sports,"4 दिसम्बर 1993 (आयु 32)कानपुर, उत्तर प्रदेश, भ...",SUPPORTS,original,,अंकित राजपूत के उपोद्घात के रूप में गेंदबाज जा...,0.236066
4,5,अंकित राजपूत,जन्म,"4 दिसम्बर 1993 (आयु 32)कानपुर, उत्तर प्रदेश, भारत","4 दिसम्बर 1985 (आयु 32)कानपुर, उत्तर प्रदेश, भारत",date,OTHER,sports,जानकारी के अनुसार अंकित राजपूत का जन्म 4 दिसम्...,REFUTES,numerical_mutation,'1993'→'1985',अंकित राजपूत के उपोद्घात के रूप में गेंदबाज जा...,0.213836


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["label"] = label_encoder.fit_transform(df["label"])

print(label_encoder.classes_)

['REFUTES' 'SUPPORTS']


In [ ]:
from sklearn.model_selection import train_test_split

# First split: 80% for training + validation, 20% for testing
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,
    random_state=42,
    stratify=train_val_df["label"]
)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize(example):
    return tokenizer(
        example["claim"],
        example["evidence"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1658 [00:00<?, ? examples/s]

Map:   0%|          | 0/237 [00:00<?, ? examples/s]

Map:   0%|          | 0/474 [00:00<?, ? examples/s]

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./nli_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
def predict_claim(claim, evidence):

    inputs = tokenizer(
        claim,
        evidence,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)

    prediction = outputs.logits.argmax().item()

    return label_encoder.inverse_transform([prediction])[0]

In [ ]:
def evaluate_nli_model(trainer, dataset, label_encoder=None):

    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        classification_report,
        confusion_matrix
    )

    print("\nRunning evaluation on dataset...\n")

    predictions = trainer.predict(dataset)

    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids

    accuracy = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted"
    )

    print("Overall Model Performance")
    print("=" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\n Classification Report")
    print("=" * 40)

    if label_encoder:
        target_names = label_encoder.classes_
        print(classification_report(
            y_true,
            y_pred,
            target_names=target_names
        ))
    else:
        print(classification_report(y_true, y_pred))

    print("\n Confusion Matrix")
    print("=" * 40)
    print(confusion_matrix(y_true, y_pred))

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.665067,0.641350,0.501209
2,No log,0.645523,0.582278,0.580567
3,0.638474,0.599628,0.611814,0.619376
4,0.638474,0.606115,0.632911,0.640303


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=832, training_loss=0.5939742418435904, metrics={'train_runtime': 403.6106, 'train_samples_per_second': 16.432, 'train_steps_per_second': 2.061, 'total_flos': 872476259573760.0, 'train_loss': 0.5939742418435904, 'epoch': 4.0})

In [ ]:
evaluate_nli_model(
    trainer,
    test_dataset,
    label_encoder
)


Running evaluation on dataset...



Overall Model Performance
Accuracy : 0.6582
Precision: 0.7026
Recall   : 0.6582
F1-score : 0.6654

 Classification Report
              precision    recall  f1-score   support

     REFUTES       0.81      0.62      0.70       305
    SUPPORTS       0.51      0.73      0.60       169

    accuracy                           0.66       474
   macro avg       0.66      0.68      0.65       474
weighted avg       0.70      0.66      0.67       474


 Confusion Matrix
[[188 117]
 [ 45 124]]


In [38]:
import pandas as pd
from sklearn.metrics import recall_score
import numpy as np

# Get predictions and true labels for the test set
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

# Create a DataFrame for analysis, ensuring indices align
# Resetting index of test_df ensures a clean alignment with y_true and y_pred
analysis_df = pd.DataFrame({
    'transformation': test_df['transformation'].reset_index(drop=True),
    'true_label': y_true,
    'predicted_label': y_pred
})

print("\nEvaluation by Transformation Type")
print("=" * 40)

# Store results
results = []

# Iterate through each unique transformation
for transformation_type in analysis_df['transformation'].unique():
    subset_df = analysis_df[analysis_df['transformation'] == transformation_type]

    count = len(subset_df)

    # Calculate recall for 'REFUTES' (label 0). 'REFUTES' is label 0 based on label_encoder.classes_ output.
    # We need to handle cases where a specific transformation might not have 'REFUTES' labels
    refutes_label_id = label_encoder.transform(['REFUTES'])[0]

    # Filter for true 'REFUTES' samples within the subset
    true_refutes_in_subset = subset_df[subset_df['true_label'] == refutes_label_id]

    recall_refutes = np.nan # Default to NaN if no 'REFUTES' samples
    if not true_refutes_in_subset.empty:
        # Calculate recall only if there are actual 'REFUTES' samples
        recall_refutes = recall_score(
            true_refutes_in_subset['true_label'],
            true_refutes_in_subset['predicted_label'],
            average='binary',
            pos_label=refutes_label_id
        )

    results.append({
        'Transformation': transformation_type,
        'Count in Test Set': count,
        'Model Recall (REFUTES)': recall_refutes
    })

# Display results as a DataFrame
results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values(by='Count in Test Set', ascending=False)
print(results_df_sorted.to_string(index=False))


Evaluation by Transformation Type
       Transformation  Count in Test Set  Model Recall (REFUTES)
             original                169                     NaN
paraphrase_corruption                146                0.945205
          entity_swap                 87                0.298851
   numerical_mutation                 41                0.365854
    semantic_contrast                 31                0.290323
